# Test `crisis_image_benchmarks_classifier` & `crisis_image_benchmarks_middleware`

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import json
import rioda_python_driver as riopy
import crisis_image_benchmarks_classifier as cibClassifier
import crisis_image_benchmarks_middleware as cibMiddleware

from tqdm import tqdm
from urllib.error import HTTPError

## 2. Test functions of classifier

In [ ]:
# Load pretrained models
pretrain = cibClassifier.load_models(verbose = False)

In [ ]:
# use informativeness task to classify image
info = cibClassifier.classify_img(
    'test_image.jpg',
    pretrain['models']['informative'],
    pretrain['decoders']['informative']
)
info

In [ ]:
# use disaster types task to classify image
disasterType = cibClassifier.classify_img(
    'test_image.jpg',
    pretrain['models']['disaster_types'],
    pretrain['decoders']['disaster_types']
)
disasterType

In [ ]:
# use humanitarian task to classify image
hum = cibClassifier.classify_img(
    'test_image.jpg',
    pretrain['models']['humanitarian'],
    pretrain['decoders']['humanitarian']
)
hum

In [ ]:
# use damage severity task to classify image
damageSeverity = cibClassifier.classify_img(
    'test_image.jpg',
    pretrain['models']['damage_severity'],
    pretrain['decoders']['damage_severity']
)
damageSeverity

In [ ]:
# use four tasks to classify image
cib = cibClassifier.cib_classify_img(
    'test_image.jpg',
    pretrain
)
cib

## 3. Test functions of middleware

In [ ]:
# load the tweets from JSON
tweets_path = '../resources/twitter-datasets/2018-aude-flood/2018-aude-flood-tweets.json'
tweets = []
for line in open(tweets_path, 'r'):
    tweets.append(json.loads(line))
print('Count of Tweets :', len(tweets))

In [ ]:
concepts = cibMiddleware.translate_labels(tweets[0], cib)
concepts

In [ ]:
concepts = []
error = 0
for tweet in tqdm(tweets):
    for media in tweet['extended_entities']['media']:
        try:
            cib = cibClassifier.cib_classify_img(media['media_url'], pretrain)
            for concept in cibMiddleware.translate_labels(tweet, cib):
                concepts.append(concept)
        except HTTPError:
            error += 1
            continue
concepts

In [ ]:
len(concepts)

In [ ]:
error